## Q8 Build your own CNN from scratch and try to achieve the highest possible accuracy
 on MNIST.

Architecture:
- Conv(32, 3x3) -> ReLU
- Conv(32, 3x3) -> ReLU
- MaxPool
- Conv(64, 3x3) -> ReLU
- Conv(64, 3x3) -> ReLU
- MaxPool
- Head: Flatten -> Dense -> ReLU -> Output

Dataset split:
- 60,000 training (MNIST train)
- 10,000 evaluation (MNIST test)

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Prefer Apple Silicon GPU (MPS), then CUDA, else CPU.
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(f"Torch version: {torch.__version__}")

torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.manual_seed_all(42)

Using device: mps
Torch version: 2.11.0


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)   # 60k
eval_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)   # 10k

batch_size = 128
pin_memory = device.type == "cuda"  # useful for CUDA; ignored for MPS/CPU

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
)
eval_loader = DataLoader(
    eval_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

print(f"Train size: {len(train_ds)}")
print(f"Eval size: {len(eval_ds)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 22.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.05MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.62MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.98MB/s]

Train size: 60000
Eval size: 10000


In [3]:
class MnistCNN(nn.Module):
    """
    Conv(32,3x3)-ReLU-Conv(32,3x3)-ReLU-MaxPool
    Conv(64,3x3)-ReLU-Conv(64,3x3)-ReLU-MaxPool
    Head: Flatten-Dense-ReLU-Output
    """

    def __init__(self, num_classes: int = 10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.head(x)


model = MnistCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

MnistCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [4]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


epochs = 5
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    eval_loss, eval_acc = evaluate(model, eval_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"eval_loss={eval_loss:.4f}, eval_acc={eval_acc:.4f}"
    )

Epoch 01 | train_loss=0.1661, train_acc=0.9468 | eval_loss=0.0494, eval_acc=0.9848
Epoch 02 | train_loss=0.0425, train_acc=0.9866 | eval_loss=0.0302, eval_acc=0.9890
Epoch 03 | train_loss=0.0282, train_acc=0.9915 | eval_loss=0.0289, eval_acc=0.9898
Epoch 04 | train_loss=0.0214, train_acc=0.9926 | eval_loss=0.0381, eval_acc=0.9879
Epoch 05 | train_loss=0.0165, train_acc=0.9949 | eval_loss=0.0225, eval_acc=0.9927


## Fine-tuning ResNet18 on Caltech 101 dataset


In [7]:
from torch.utils.data import random_split

# ImageNet-style normalization works well for transfer learning models like ResNet.
# Some Caltech101 images are grayscale, so force 3 channels first.
caltech_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

caltech_full = datasets.Caltech101(
    root="./data",
    download=True,
    transform=caltech_transform,
)

n_total = len(caltech_full)
n_train = int(0.8 * n_total)
n_test = n_total - n_train

# Reproducible split
split_gen = torch.Generator().manual_seed(42)
caltech_train_ds, caltech_test_ds = random_split(
    caltech_full,
    [n_train, n_test],
    generator=split_gen,
)

batch_size = 64
pin_memory = device.type == "cuda"

caltech_train_loader = DataLoader(
    caltech_train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
)

caltech_test_loader = DataLoader(
    caltech_test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

print(f"Caltech101 total: {n_total}")
print(f"Train (80%): {len(caltech_train_ds)}")
print(f"Test (20%): {len(caltech_test_ds)}")

Caltech101 total: 8677
Train (80%): 6941
Test (20%): 1736


In [ ]:
from torchvision import models

# Build ResNet18 and fine-tune only the output layer.
num_classes = len(caltech_full.categories)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in resnet.parameters():
    param.requires_grad = False

in_features = resnet.fc.in_features
resnet.fc = nn.Linear(in_features, num_classes)
resnet = resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet.fc.parameters(), lr=1e-3)


def train_one_epoch_head_only(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_accuracy(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


epochs = 15
best_eval_acc = 0.0

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch_head_only(
        resnet, caltech_train_loader, optimizer, criterion, device
    )
    eval_loss, eval_acc = evaluate_accuracy(
        resnet, caltech_test_loader, criterion, device
    )

    best_eval_acc = max(best_eval_acc, eval_acc)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"eval_loss={eval_loss:.4f}, eval_acc={eval_acc:.4f}"
    )

print(f"Best eval accuracy over {epochs} epochs: {best_eval_acc:.4f}")

Epoch 01 | train_loss=2.4246, train_acc=0.5176 | eval_loss=1.2423, eval_acc=0.7615
Epoch 02 | train_loss=0.8604, train_acc=0.8571 | eval_loss=0.7528, eval_acc=0.8548
Epoch 03 | train_loss=0.5327, train_acc=0.9150 | eval_loss=0.5936, eval_acc=0.8802
Epoch 04 | train_loss=0.3970, train_acc=0.9323 | eval_loss=0.5178, eval_acc=0.8819
Epoch 05 | train_loss=0.3122, train_acc=0.9478 | eval_loss=0.4715, eval_acc=0.8865
Epoch 06 | train_loss=0.2565, train_acc=0.9587 | eval_loss=0.4450, eval_acc=0.8975


In [ ]:
# Fine-tune ResNet18 with only layer4 + output layer trainable.
resnet_l4 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze everything first.
for param in resnet_l4.parameters():
    param.requires_grad = False

# Unfreeze layer4 block.
for param in resnet_l4.layer4.parameters():
    param.requires_grad = True

# Replace and train output layer.
in_features = resnet_l4.fc.in_features
resnet_l4.fc = nn.Linear(in_features, num_classes)
for param in resnet_l4.fc.parameters():
    param.requires_grad = True

resnet_l4 = resnet_l4.to(device)

criterion_l4 = nn.CrossEntropyLoss()
optimizer_l4 = torch.optim.Adam(
    (p for p in resnet_l4.parameters() if p.requires_grad),
    lr=1e-4,
)

trainable_params = sum(p.numel() for p in resnet_l4.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in resnet_l4.parameters())
print(f"Trainable params: {trainable_params:,} / {total_params:,}")


def train_one_epoch_l4(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_l4(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


epochs = 15
best_eval_acc_l4 = 0.0

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch_l4(
        resnet_l4, caltech_train_loader, optimizer_l4, criterion_l4, device
    )
    eval_loss, eval_acc = evaluate_l4(
        resnet_l4, caltech_test_loader, criterion_l4, device
    )

    best_eval_acc_l4 = max(best_eval_acc_l4, eval_acc)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"eval_loss={eval_loss:.4f}, eval_acc={eval_acc:.4f}"
    )

print(f"Best eval accuracy (layer4 + fc): {best_eval_acc_l4:.4f}")

Trainable params: 8,445,541 / 11,228,325
Epoch 01 | train_loss=2.0989, train_acc=0.6027 | eval_loss=1.0281, eval_acc=0.8168
Epoch 02 | train_loss=0.6208, train_acc=0.9213 | eval_loss=0.5416, eval_acc=0.9199
Epoch 03 | train_loss=0.2435, train_acc=0.9808 | eval_loss=0.3886, eval_acc=0.9401
Epoch 04 | train_loss=0.1094, train_acc=0.9950 | eval_loss=0.3205, eval_acc=0.9430
Epoch 05 | train_loss=0.0544, train_acc=0.9990 | eval_loss=0.2989, eval_acc=0.9482


In [ ]:
# Quick debug: list all trainable parameters.
for name, param in resnet_l4.named_parameters():
    if param.requires_grad:
        print(name)